# Stage 2: the floor

The variance of the partial estimator splits exactly into a sampling term, which falls like `1/m`, and an `m`-independent floor `Var_nu(O)`:

    Var(O_hat_m) = (n-m)/(m(n-1)) * E[V_x] + Var_nu(O).

For the shuffle both terms are known in closed form. For critical systems they are measured.

In [1]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from rsi_shuffle import theory
print(" n      R_n          n*R_n")
for n in (10, 100, 1000, 10_000, 100_000):
    R = theory.relative_floor(n)
    print(f"{n:>7} {R:.4e}   {n*R:.4f}")
print("limit of n*R_n: 2/5 = 0.4")

 n      R_n          n*R_n
     10 4.6465e-02   0.4646
    100 4.0422e-03   0.4042
   1000 4.0040e-04   0.4004
  10000 4.0004e-05   0.4000
 100000 4.0000e-06   0.4000
limit of n*R_n: 2/5 = 0.4


## Where the floor starts to bind

For `n = 1024` the squared coefficient of variation follows `1/(2m)` until `m` is a sizeable fraction of `n`, and then flattens onto `R_n`.

In [2]:
n = 1024
for m in (8, 64, 256, 512, 1024):
    cv2 = theory.exact_cv2(n, m)
    print(f"m={m:>5}  CV^2 = {cv2:.3e}   1/(2m) = {1/(2*m):.3e}   floor share = {theory.relative_floor(n)/cv2:.2f}")

m=    8  CV^2 = 6.241e-02   1/(2m) = 6.250e-02   floor share = 0.01
m=   64  CV^2 = 7.717e-03   1/(2m) = 7.812e-03   floor share = 0.05
m=  256  CV^2 = 1.856e-03   1/(2m) = 1.953e-03   floor share = 0.21
m=  512  CV^2 = 8.794e-04   1/(2m) = 9.766e-04   floor share = 0.44
m= 1024  CV^2 = 3.910e-04   1/(2m) = 4.883e-04   floor share = 1.00


## Certification cost

To certify that the mean has reached equilibrium to relative tolerance `delta` (size 0.05, power 0.80) one needs `m*(delta) = kappa^2 / (rho^2 (delta^2 - kappa^2 R_n))`, with `rho^2 = 2` for the shuffle. Below `delta = kappa sqrt(R_n)` no sample is enough.

The formula is leading order in `m/n`. A value of `m*` at or above `n` (here `n = 256` at `delta = 0.10`) means the leading-order cost is outside its range, which for this `n` is the same statement as `delta` lying close to the impossibility threshold.

In [3]:
kappa = theory.kappa()
for n in (256, 1024, 10**5):
    thr = kappa*np.sqrt(theory.relative_floor(n))
    print(f"n={n:>7}: impossible below delta = {thr:.4f};  m*(0.10) = {theory.certification_cost(0.10, n):.0f}")

n=    256: impossible below delta = 0.0985;  m*(0.10) = 10263
n=   1024: impossible below delta = 0.0492;  m*(0.10) = 408
n= 100000: impossible below delta = 0.0050;  m*(0.10) = 310


## The critical Ising model

Wolff-sampler runs at `T_c` (regenerated for this repository, see `data/extension/regenerated`). The energy density has a floor that shrinks with `n`; the magnitude of the magnetisation has one that does not.

In [4]:
ising = pd.read_csv("../data/extension/regenerated/ising_critical_sizes.csv")
print(ising[["L","R_M","R_M_se","R_E","R_E_se","chi_M","gate_pass"]].round(5).to_string(index=False))

  L     R_M  R_M_se     R_E  R_E_se     chi_M  gate_pass
  8 0.07070 0.00025 0.04153 0.00009   2.73355       True
 12 0.07073 0.00026 0.02237 0.00005   5.57608       True
 16 0.07145 0.00028 0.01430 0.00004   9.31060       True
 24 0.07126 0.00027 0.00730 0.00002  18.91870       True
 32 0.07232 0.00028 0.00454 0.00001  31.69680       True
 48 0.07088 0.00040 0.00223 0.00001  63.41910       True
 64 0.07252 0.00047 0.00137 0.00001 106.93400       True
 96 0.07255 0.00066 0.00066 0.00000 216.56600       True
128 0.07239 0.00073 0.00040 0.00000 358.67800       True


In [5]:
import json
s = json.load(open("../data/extension/regenerated/ising_critical_slopes.json"))
for k, v in s["slopes_vs_log_n"].items():
    lo, hi = s["bootstrap_95ci"][k]
    print(f"slope of {k:5s} vs log n: {v:+.3f}   95% interval [{lo:+.3f}, {hi:+.3f}]")
print("universal value of the susceptibility exponent: gamma/(2 nu) = 7/8 = 0.875")

slope of R_M   vs log n: +0.004   95% interval [+0.000, +0.008]
slope of R_E   vs log n: -0.863   95% interval [-0.866, -0.860]
slope of chi_M vs log n: +0.878   95% interval [+0.875, +0.881]
universal value of the susceptibility exponent: gamma/(2 nu) = 7/8 = 0.875


The exact energy floor from Kaufman's partition function has an effective slope of -0.864 over the same sizes (`results/outputs/extension-verification/ising_exact.txt`), which is what the Monte Carlo slope should be compared with.

## Mean field

Curie-Weiss, summed exactly: `Var(M)` falls like `1/N` above `T_c` and stays constant below it.

In [6]:
cw = pd.read_csv("../data/extension/regenerated/curie_weiss_exact.csv")
for T, g in cw.groupby("T"):
    g = g[(g.N >= 100) & (g.N <= 3000)]
    print(f"T = {T:>4}: slope of log Var(M) vs log N = {np.polyfit(np.log(g.N), np.log(g.var_M), 1)[0]:+.3f}")

T =  0.7: slope of log Var(M) vs log N = +0.006
T =  0.9: slope of log Var(M) vs log N = +0.057
T =  1.0: slope of log Var(M) vs log N = -0.495
T =  1.5: slope of log Var(M) vs log N = -0.986
T =  2.0: slope of log Var(M) vs log N = -0.995
